In [1]:
import os
import shutil
from google.colab import drive
drive.mount('/content/drive')

# Define model name and paths
model_name = "gemma4-e4b" # change this
drive_folder = "/content/drive/MyDrive/Models/finetunes/" + model_name + "/"
drive_gguf_path = drive_folder + "gemma-4-e4b-it.Q8_0" + ".gguf" # and this
dataset = "gad7_valid_dataset.json" # and this
drive_dataset_path = "/content/drive/MyDrive/Models/" + dataset

local_gguf_path = "/content/" + model_name + ".gguf"
local_dataset_path = "/content/" + dataset

# Copy GGUF file to local runtime
print(f"Copying model from Google Drive to local runtime...")
if not os.path.exists(local_gguf_path):
    shutil.copy2(drive_gguf_path, local_gguf_path)

# Copy dataset to local runtime
print(f"Copying dataset from Google Drive to local runtime...")
if os.path.exists(drive_dataset_path):
    shutil.copy2(drive_dataset_path, local_dataset_path)
else:
    print("Warning: Dataset not found in Drive directory.")

print("Copy complete!")

Mounted at /content/drive
Copying model from Google Drive to local runtime...
Copying dataset from Google Drive to local runtime...
Copy complete!


In [2]:
!sudo apt update
!sudo apt install -y pciutils zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [93.4 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,004 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,251 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,247 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-

In [3]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [4]:
# Create the Modelfile pointing to the local file
modelfile_content = f"FROM {local_gguf_path}\n"
with open("Modelfile", "w") as f:
    f.write(modelfile_content)

# Create the model in Ollama
!ollama create {model_name} -f /content/Modelfile

# List available models to verify
!ollama list


NAME                 ID              SIZE      MODIFIED               
gemma4-e4b:latest    6b0099b44027    8.0 GB    Less than a second ago    


In [5]:
%pip install openai scikit-learn pandas

In [6]:
# Create client

from openai import OpenAI
from sklearn.model_selection import train_test_split
import pandas as pd

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",  # Ollama
    api_key="sk-1234"  # Dummy key
)

In [7]:
SYSTEM_PROMPT = """
You are a GAD-7 survey scoring assistant.
Your job is to extract answers for the GAD-7 based on a given conversation transcript.
The Generalized Anxiety Disorder 7-item scale (GAD-7) is a clinical assessment tool that measures the severity of anxiety symptoms over the past two weeks.

The conversation asks questions in this specific order. Map each user response to the correct q number:
q1   Feeling nervous, anxious, or on edge
q2   Not being able to stop or control worrying
q3   Worrying too much about different things
q4   Trouble relaxing
q5   Being so restless that it's hard to sit still
q6   Becoming easily annoyed or irritable
q7   Feeling afraid, as if something awful might happen

After q7, the assistant asks a follow-up about how much these feelings have interfered with daily life (work, home, relationships). This is the functional impairment question and is NOT scored 0-3 — it is recorded as one of these four exact strings:
  "Not difficult at all"
  "Somewhat difficult"
  "Very difficult"
  "Extremely difficult"

Scale mapping for q1-q7:
0 = not at all
1 = several days
2 = more than half the days
3 = nearly every day

Input Format:
You will take input in the form of a transcript in JSON matching OpenAI's API format.
Example:
[
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "..."},
    {"role": "user", "content": "..."},
    // ...
]

Output Format:
- For q1 through q7, return a single string value: "0", "1", "2", or "3".
- For functional_impairment, return one of the four exact strings listed above.
Rules:
- Match each topic to the correct q number based on the descriptions above. The interviewer may ask multiple follow-up questions for the same item; treat the user's overall answer for that topic as the value.
- Map natural-language frequencies to the closest scale value (e.g., "not at all" / "never" -> 0, "a few days" / "once or twice" / "several days" -> 1, "more than half the days" / "most days" -> 2, "every day" / "nearly every day" / "constantly" / "all the time" -> 3).
- Map the final functional impairment answer to the closest of the four labels (e.g., a user saying "it's been quite a bit, very difficult" -> "Very difficult"; "barely affects me" -> "Not difficult at all"; "I'm drowning" / "completely overwhelmed" -> "Extremely difficult").
- Score each item based solely on what the user said.
- Return ONLY valid JSON.
- Do not include markdown fences.
- Do not include explanation.
- Do not omit any key.
Return JSON in exactly this shape:
{
  "q1": "0",
  "q2": "0",
  "q3": "0",
  "q4": "0",
  "q5": "0",
  "q6": "0",
  "q7": "0",
  "functional_impairment": "Not difficult at all"
}
"""
MODEL = model_name

- Use a json dataset of GAD-7 conversation transcripts
- Extract scores using LLM
- Validate whether they were correct using MSE on q1-q7 and accuracy on functional_impairment

In [8]:
# Load the GAD-7 dataset
import json

with open('gad7_valid_dataset.json') as f:
    data = json.loads('\n'.join([line for line in f]))

# Split the data into conversation data and expected values
conversations = [entry['conversation'] for entry in data]
expected_outputs = [entry['expected_output'] for entry in data]

print('SUCCESS: Number of conversations matches number of scores' if len(conversations) == len(expected_outputs) else 'FAIL: Number of conversations does not match number of scores')
print(f'Total conversations: {len(conversations)}')

SUCCESS: Number of conversations matches number of scores
Total conversations: 10


In [9]:
# Further split the data into training and test data
conv_train, _conv_test, score_train, _score_test = train_test_split(conversations, expected_outputs, random_state=1234)
print(f'Train: {len(conv_train)}, Test: {len(_conv_test)}')

Train: 7, Test: 3


In [10]:
def Message(content, role='user'):
    return {
        'role': role,
        'content': content
    }

def create_prediction(transcript) -> str:
    completion = client.chat.completions.create(
        model=MODEL,
        temperature=0.1,  # low temperature for consistent scoring
        messages=[
            Message(SYSTEM_PROMPT, 'system'),
            Message(json.dumps(transcript))
        ],
    )
    return completion.choices[0].message.content

In [11]:
# Run predictions on training and test set
def generate_scores(conversations):
    scores = []
    for i, c in enumerate(conversations):
        print(f'Generating {i+1}/{len(conversations)}')
        scores.append(create_prediction(c))
    return scores

train_scores_raw = generate_scores(conv_train)
test_scores_raw = generate_scores(_conv_test)

Generating 1/7
Generating 2/7
Generating 3/7
Generating 4/7
Generating 5/7
Generating 6/7
Generating 7/7
Generating 1/3
Generating 2/3
Generating 3/3


In [12]:
# Parse JSON scores
import json

def parse_scores(raw_scores):
    parsed = []
    default = {f'q{i}': '0' for i in range(1, 8)}
    default['functional_impairment'] = 'Not difficult at all'
    for s in raw_scores:
        try:
            parsed.append(json.loads(s))
        except json.JSONDecodeError:
            print(f'Failed to parse: {s}')
            parsed.append(dict(default))
    return parsed

train_scores_parsed = parse_scores(train_scores_raw)
test_scores_parsed = parse_scores(test_scores_raw)

In [13]:
from sklearn.metrics import mean_squared_error
import pandas as pd

def convert_scores_to_array(scores_df):
    return [int(s) if s else 0 for s in scores_df.values.flatten().tolist()]

def evaluate_split(parsed_scores, expected_scores, name):
    pred_df = pd.DataFrame(parsed_scores)
    expected_df = pd.DataFrame(expected_scores)

    # Numeric items q1-q7 for MSE
    num_cols = [f'q{i}' for i in range(1, 8)]
    pred_num = pred_df[num_cols]
    expected_num = expected_df[num_cols]

    mse = mean_squared_error(
        convert_scores_to_array(expected_num),
        convert_scores_to_array(pred_num)
    )
    print(f'{name} MSE (q1-q7): {mse}')

    # Functional impairment accuracy
    fi_correct = (pred_df['functional_impairment'] == expected_df['functional_impairment']).sum()
    print(f'{name} Functional impairment accuracy: {fi_correct}/{len(expected_df)}')

    # Combined diff view
    all_cols = num_cols + ['functional_impairment']
    diff_df = expected_df[all_cols].compare(pred_df[all_cols], align_axis=1, keep_shape=True, keep_equal=True).rename(
        columns={'self': 'Expected', 'other': 'Predicted'}, level=1
    )
    display(diff_df)

evaluate_split(train_scores_parsed, score_train, 'Train')
evaluate_split(test_scores_parsed, _score_test, 'Test')

Train MSE (q1-q7): 0.16326530612244897
Train Functional impairment accuracy: 7/7


q1                 q2                 q3                 q4            \
  Expected Predicted Expected Predicted Expected Predicted Expected Predicted   
0        2         2        1         2        2         2        2         2   
1        1         1        0         0        1         1        1         1   
2        3         3        3         3        3         3        3         3   
3        3         2        2         2        3         2        2         2   
4        2         2        1         2        1         2        2         2   
5        3         3        3         3        3         3        3         3   
6        3         3        3         3        3         3        3         3   

        q5                 q6                 q7            \
  Expected Predicted Expected Predicted Expected Predicted   
0        0         1        1         2        1         1   
1        0         0        0         1        0         0   
2        2         2        3         3        3         3   
3        1         1        2         2        1         1   
4        1         1        2         2        1         1   
5        1         1        3         3        3         3   
6        3         3        3         3        3         3   

  functional_impairment                        
               Expected             Predicted  
0    Somewhat difficult    Somewhat difficult  
1  Not difficult at all  Not difficult at all  
2   Extremely difficult   Extremely difficult  
3        Very difficult        Very difficult  
4        Very difficult        Very difficult  
5   Extremely difficult   Extremely difficult  
6   Extremely difficult   Extremely difficult

Test MSE (q1-q7): 0.14285714285714285
Test Functional impairment accuracy: 3/3


q1                 q2                 q3                 q4            \
  Expected Predicted Expected Predicted Expected Predicted Expected Predicted   
0        3         2        2         2        1         2        2         2   
1        3         3        2         2        3         3        3         3   
2        3         3        3         3        3         3        3         3   

        q5                 q6                 q7            \
  Expected Predicted Expected Predicted Expected Predicted   
0        1         1        1         1        1         1   
1        1         1        2         2        2         1   
2        1         1        3         3        2         2   

  functional_impairment                      
               Expected           Predicted  
0    Somewhat difficult  Somewhat difficult  
1        Very difficult      Very difficult  
2        Very difficult      Very difficult